In [20]:
import numpy as np
import pandas as pd
from pathlib import Path
path = Path.cwd().parents[1]

rng = np.random.default_rng(42)

TRAIN_PATH = "data/raw/mnist_train.npz"
TEST_PATH = "data/raw/mnist_test.npz"
train_data = np.load(path / TRAIN_PATH)
test_data = np.load(path / TEST_PATH)

X_train, y_train = train_data["X"], train_data["y"]
X_test, y_test = test_data["X"], test_data["y"]

In [21]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]   # src/models -> Classify_Number
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [22]:
from src.preprocessing.scaler import prepare

data = prepare("standard", X_train, y_train, X_test, y_test, val_size=0.2)
X_train = data['X_train']
X_test  = data['X_test']
X_val   = data['X_val']
y_train = data['y_train']
y_test  = data['y_test']
y_val   = data['y_val']
tool = data['tools']
len(data['X_train'][0])

784

In [23]:
import numpy as np
import torch
from sklearn.model_selection import StratifiedKFold
from mlp import MLP


def kfold_evaluate(X, y, k=5, sizes=(784, 128, 10), epochs=20, seed=0):
    """X: np.ndarray (N, 784) đã chuẩn hóa về [0, 1]; y: np.ndarray (N,)."""
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
    accs = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        X_tr = torch.tensor(X[tr_idx], dtype=torch.float32)
        y_tr = torch.tensor(y[tr_idx], dtype=torch.long)
        X_va = torch.tensor(X[va_idx], dtype=torch.float32)
        y_va = torch.tensor(y[va_idx], dtype=torch.long)

        model = MLP(list(sizes), seed=seed + fold)
        model.fit(X_tr, y_tr, epochs=epochs, batch_size=128, lr=0.05,
                  momentum=0.9, seed=seed + fold, verbose=False)

        acc = (model.predict(X_va) == y_va).float().mean().item()
        accs.append(acc)
        print(f"Fold {fold}: acc = {acc:.4f}")

    accs = np.array(accs)
    print(f"\nTrung bình: {accs.mean():.4f} ± {accs.std(ddof=1):.4f}")
    return accs

In [24]:
kfold_evaluate(X_train, y_train)

Fold 1: acc = 0.9698
Fold 2: acc = 0.9693
Fold 3: acc = 0.9731
Fold 4: acc = 0.9706
Fold 5: acc = 0.9694

Trung bình: 0.9704 ± 0.0016


array([0.96979165, 0.96927083, 0.97312498, 0.97062498, 0.96937501])